# Interval Decoding

This notebook contains plotting only. Both decoders save result tables, and all visualization now happens here.


In [ ]:
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [ ]:
# Publication styling: register a compact 'pub' template that all figures
# below inherit via template='plotly_white+pub'. Keeps a consistent
# Nature-style look (Arial, small fonts, thin colorbars) so that only
# per-figure width has to be tuned.
import plotly.io as pio

pio.templates['pub'] = go.layout.Template(
    layout=dict(
        font=dict(family='Arial, Helvetica, sans-serif', size=9, color='black'),
        title=dict(font=dict(size=12, color='black'), x=0.02, xanchor='left'),
        paper_bgcolor='white',
        plot_bgcolor='white',
        legend=dict(font=dict(size=8)),
        coloraxis=dict(colorbar=dict(
            thickness=10,
            len=0.6,
            title=dict(font=dict(size=9)),
            tickfont=dict(size=8),
        )),
    )
)


In [ ]:
import re

TARGET_ORDER = ['Cue', 'Outcome', 'R1 choice', 'R2 choice']

BIN_RUN_PREFIXES = [
    'run_logreg',
]
SESSION_RUN_PREFIXES = [
    'log_reg_5k_3intervals',
]
ASSEMBLIES = None  # None -> auto-discover all Assembly### runs, or set e.g. ['Assembly012']
BALANCED_ACCURACY_INTERVALS = [
    'cue_entry_interval',
    'R1_entry_interval',
    'R2_entry_interval',
]  # Set to None to show all intervals in balanced-accuracy plots.

print('Train command examples:')
print('python scripted_plotting/animal_6_analysis/interval_decoding_train.py --run-name run_logreg --assembly-col all --n-jobs 4')
print('python scripted_plotting/animal_6_analysis/interval_decoding_train.py --run-name run_logreg --assembly-col all --n-jobs 4 --bootstrap-n 1000 --shuffle-n 1000')
print('python scripted_plotting/animal_6_analysis/interval_session_decoding_train.py --run-name log_reg_5k_3intervals --assembly-col all --shuffle-n 5000 --interval-name cue_entry_interval --interval-name R1_entry_interval --interval-name R2_entry_interval')
print('Session-level parallel example (manual per-assembly fan-out; no --n-jobs in current script):')
print("printf '%s\\n' Assembly001 Assembly002 Assembly003 Assembly004 | xargs -I{} -P 4 python scripted_plotting/animal_6_analysis/interval_session_decoding_train.py --assembly-col {} --run-name log_reg_5k_3intervals_{} --shuffle-n 5000 --interval-name cue_entry_interval --interval-name R1_entry_interval --interval-name R2_entry_interval")


def _find_runs_base(candidate_rels) -> Path:
    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents]:
        for rel in candidate_rels:
            cand = root / rel
            if cand.exists():
                return cand
    raise FileNotFoundError(f'Could not locate any run directory from {candidate_rels}')


def _read_table(run_dir: Path, stem: str) -> pd.DataFrame:
    parquet_path = run_dir / f'{stem}.parquet'
    csv_path = run_dir / f'{stem}.csv'
    if parquet_path.exists():
        try:
            return pd.read_parquet(parquet_path)
        except Exception:
            if not csv_path.exists():
                raise
    if csv_path.exists():
        return pd.read_csv(csv_path)
    return pd.DataFrame()


def _read_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text())


def _assembly_sort_key(name: str):
    match = re.fullmatch(r'Assembly(\d+)', str(name))
    if match:
        return (0, int(match.group(1)))
    return (1, str(name))


def _discover_assembly_names(runs_base: Path, run_prefixes) -> list[str]:
    assemblies = set()
    for prefix in run_prefixes:
        pattern = re.compile(re.escape(prefix) + r'_+(Assembly\d+)$')
        for child in runs_base.iterdir():
            if not child.is_dir():
                continue
            match = pattern.fullmatch(child.name)
            if match:
                assemblies.add(match.group(1))
    return sorted(assemblies, key=_assembly_sort_key)


def _find_matching_run_name(runs_base: Path, prefix: str, assembly: str) -> str | None:
    pattern = re.compile(re.escape(prefix) + r'_+' + re.escape(str(assembly)) + r'$')
    matches = sorted(
        child.name
        for child in runs_base.iterdir()
        if child.is_dir() and pattern.fullmatch(child.name)
    )
    return matches[0] if matches else None


def build_run_names(run_prefixes, candidate_rels, assemblies=None):
    runs_base = _find_runs_base(candidate_rels)
    selected_assemblies = list(assemblies) if assemblies else _discover_assembly_names(runs_base, run_prefixes)
    run_names = []
    for prefix in run_prefixes:
        if selected_assemblies:
            for assembly in selected_assemblies:
                match_name = _find_matching_run_name(runs_base, prefix, assembly)
                if match_name is not None:
                    run_names.append(match_name)
        elif (runs_base / prefix).exists():
            run_names.append(prefix)
    return run_names


def _bh_fdr(p_values) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    q_values = np.full(p_values.shape, np.nan, dtype=float)
    valid = np.isfinite(p_values)
    if not valid.any():
        return q_values

    p_valid = p_values[valid]
    order = np.argsort(p_valid)
    ranked = p_valid[order]
    scale = ranked.size / np.arange(1, ranked.size + 1, dtype=float)
    q_ranked = np.minimum.accumulate((ranked * scale)[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0.0, 1.0)

    q_valid = np.empty_like(p_valid)
    q_valid[order] = q_ranked
    q_values[valid] = q_valid
    return q_values


def _add_shuffle_fdr_columns(results_df: pd.DataFrame) -> pd.DataFrame:
    out = results_df.copy()
    if 'p_shuffle' not in out.columns:
        out['p_shuffle_fdr_bh'] = np.nan
        out['significant_shuffle_fdr_bh_0_05'] = np.nan
        return out

    q_values = _bh_fdr(pd.to_numeric(out['p_shuffle'], errors='coerce').to_numpy(dtype=float))
    out['p_shuffle_fdr_bh'] = q_values

    # This function is intentionally reusable: when called on a run-level table it
    # gives run-level FDR; when called on a filtered plotting table it gives
    # figure-scoped FDR for exactly the rows passed in.
    significant = pd.Series(np.nan, index=out.index, dtype=object)
    valid = np.isfinite(q_values)
    significant.loc[valid] = q_values[valid] < 0.05
    out['significant_shuffle_fdr_bh_0_05'] = significant
    return out


def load_run_bundle(run_names, candidate_rels):
    empty = {
        'base_dir': None,
        'run_dirs': [],
        'results_df': pd.DataFrame(),
        'diagnostics_df': pd.DataFrame(),
        'meta_df': pd.DataFrame(),
        'config_df': pd.DataFrame(),
    }
    if not run_names:
        return empty

    runs_base = _find_runs_base(candidate_rels)
    run_dirs = [runs_base / name for name in run_names if (runs_base / name).exists()]

    results_parts = []
    diagnostics_parts = []
    meta_by_run = {}
    config_by_run = {}
    for run_dir in run_dirs:
        results_df_run = _read_table(run_dir, 'results')
        if results_df_run.empty:
            continue

        results_df_run['run_name'] = run_dir.name
        results_df_run = _add_shuffle_fdr_columns(results_df_run)
        results_parts.append(results_df_run)

        diagnostics_df_run = _read_table(run_dir, 'diagnostics')
        if not diagnostics_df_run.empty:
            diagnostics_df_run['run_name'] = run_dir.name
            diagnostics_parts.append(diagnostics_df_run)

        meta_payload = _read_json(run_dir / 'meta.json')
        if meta_payload is not None:
            meta_by_run[run_dir.name] = meta_payload

        config_payload = _read_json(run_dir / 'config.json')
        if config_payload is not None:
            config_by_run[run_dir.name] = config_payload

    return {
        'base_dir': runs_base,
        'run_dirs': run_dirs,
        'results_df': pd.concat(results_parts, ignore_index=True) if results_parts else pd.DataFrame(),
        'diagnostics_df': pd.concat(diagnostics_parts, ignore_index=True) if diagnostics_parts else pd.DataFrame(),
        'meta_df': pd.DataFrame(meta_by_run).T if meta_by_run else pd.DataFrame(),
        'config_df': pd.DataFrame(config_by_run).T if config_by_run else pd.DataFrame(),
    }


def _target_order(values):
    present = [target for target in TARGET_ORDER if target in set(values)]
    present += sorted(set(values) - set(present))
    return present


def _resolve_interval_order(values, interval_names=None):
    present = list(dict.fromkeys(pd.Series(values).astype(str)))
    if interval_names is None:
        return present
    requested = [str(interval_name) for interval_name in interval_names]
    present_set = set(present)
    return [interval_name for interval_name in requested if interval_name in present_set]


def _filter_interval_rows(df: pd.DataFrame, interval_names=None):
    interval_order = _resolve_interval_order(df['interval_name'], interval_names)
    if interval_names is None:
        return df.copy(), interval_order
    if not interval_order:
        return df.iloc[0:0].copy(), interval_order
    filtered = df[df['interval_name'].astype(str).isin(interval_order)].copy()
    return filtered, interval_order


def _parse_session_date_token(session_name: str):
    for token in str(session_name).split('_'):
        try:
            return datetime.strptime(token, '%Y-%m-%d')
        except ValueError:
            continue
    return None


def _session_sort_key(session_name: str):
    parsed = _parse_session_date_token(session_name)
    fallback = datetime.max if parsed is None else parsed
    return fallback, str(session_name)


def _preview_runs(run_names):
    if len(run_names) <= 6:
        return run_names
    return [*run_names[:3], '...', *run_names[-3:]]


BIN_RUN_NAMES = build_run_names(
    BIN_RUN_PREFIXES,
    [
        'scripted_plotting/animal_6_analysis/interval_decoding_runs',
        'interval_decoding_runs',
    ],
    assemblies=ASSEMBLIES,
)
SESSION_RUN_NAMES = build_run_names(
    SESSION_RUN_PREFIXES,
    [
        'scripted_plotting/animal_6_analysis/interval_session_decoding_runs',
        'interval_session_decoding_runs',
    ],
    assemblies=ASSEMBLIES,
)

print(f'Configured bin runs ({len(BIN_RUN_NAMES)}):', _preview_runs(BIN_RUN_NAMES))
print(f'Configured session runs ({len(SESSION_RUN_NAMES)}):', _preview_runs(SESSION_RUN_NAMES))



## Bin-Wise Decoder Runs


In [ ]:
bin_bundle = load_run_bundle(
    BIN_RUN_NAMES,
    [
        'scripted_plotting/animal_6_analysis/interval_decoding_runs',
        'interval_decoding_runs',
    ],
)
bin_results_df = bin_bundle['results_df']
bin_diagnostics_df = bin_bundle['diagnostics_df']

if bin_results_df.empty:
    print('No bin-wise runs loaded.')
else:
    print(f"Loaded bin-wise runs: {[run_dir.name for run_dir in bin_bundle['run_dirs']]}")
    display(bin_bundle['meta_df'])
    display(bin_bundle['config_df'])
    display(bin_results_df.head())
    display(bin_diagnostics_df.head())


### Best Bins


In [ ]:
bin_summary_df = (
    bin_results_df[bin_results_df['balanced_accuracy'].notna()]
    .sort_values(
        ['run_name', 'target_name', 'interval_name', 'balanced_accuracy', 'bootstrap_mean'],
        ascending=[True, True, True, False, False],
        na_position='last',
    )
    .groupby(['run_name', 'target_name', 'interval_name'], as_index=False)
    .first()
)

bin_summary_cols = [
    'run_name', 'target_name', 'interval_name', 'model_bin', 'balanced_accuracy',
    'accuracy', 'p_above_chance', 'p_shuffle', 'bootstrap_ci_low', 'bootstrap_ci_high'
]
if bin_summary_df.empty:
    print('No bin-wise summary rows available.')
else:
    display(bin_summary_df[bin_summary_cols].sort_values(['run_name', 'target_name', 'interval_name']))


## Session-Level Decoder Runs

Each decoder is fit separately inside each session using all bins from an interval as one per-trial feature vector.


In [ ]:
session_bundle = load_run_bundle(
    SESSION_RUN_NAMES,
    [
        'scripted_plotting/animal_6_analysis/interval_session_decoding_runs',
        'interval_session_decoding_runs',
    ],
)
session_results_df = session_bundle['results_df']
session_diagnostics_df = session_bundle['diagnostics_df']

if session_results_df.empty:
    print('No session-level runs loaded.')
else:
    print(f"Loaded session-level runs: {[run_dir.name for run_dir in session_bundle['run_dirs']]}")
    if not session_bundle['meta_df'].empty:
        display(session_bundle['meta_df'])
    if not session_bundle['config_df'].empty:
        display(session_bundle['config_df'])
    display(session_results_df.head())
    if not session_diagnostics_df.empty:
        display(session_diagnostics_df.head())


In [ ]:
def _coerce_bool_flag(raw) -> bool:
    return bool(raw) if pd.notna(raw) else False


def plot_session_interval_heatmaps(
    results_df: pd.DataFrame,
    value_col: str = 'balanced_accuracy',
    title_prefix: str = 'Session-level decoding',
    extra_cols=None,
    ncols: int = 3,
    colorscale: str = 'RdBu_r',
    zmid: float = 0.5,
    zmin=None,
    zmax=None,
    significance_col=None,
    significance_label: str = 'shuffle significant (p<0.05)',
    min_trials=None,
    significance_marker_size: float = 7.0,
    interval_names=None,
    excluded_targets=('Outcome',),
):
    if extra_cols is None:
        extra_cols = []

    if results_df.empty:
        print('No session-level results loaded.')
        return
    if value_col not in results_df.columns:
        print(f"Column '{value_col}' is not available in the loaded session results.")
        return

    excluded_targets_lower = {str(target).lower() for target in excluded_targets}
    extra_cols = [extra_col for extra_col in extra_cols if extra_col in results_df.columns]
    valid = results_df[results_df[value_col].notna()].copy()
    valid = valid[~valid['target_name'].astype(str).str.lower().isin(excluded_targets_lower)].copy()
    if min_trials is not None and 'n_trials' in valid.columns:
        valid = valid[pd.to_numeric(valid['n_trials'], errors='coerce') >= float(min_trials)].copy()
    if valid.empty:
        trial_suffix = f' after filtering to n_trials >= {min_trials}' if min_trials is not None else ''
        print(f'No rows with {value_col}{trial_suffix} after excluding {sorted(excluded_targets)}.')
        return

    separator_date = datetime.strptime('2025-01-23', '%Y-%m-%d')

    for run_name, run_df in valid.groupby('run_name', sort=False):
        run_df, intervals = _filter_interval_rows(run_df, interval_names)
        if run_df.empty or not intervals:
            continue
        run_df = _add_shuffle_fdr_columns(run_df)
        targets = _target_order(run_df['target_name'].astype(str))
        all_sessions = sorted(run_df['session_id'].astype(str).unique().tolist(), key=_session_sort_key)

        nrows = int(np.ceil(len(intervals) / ncols))
        fig = make_subplots(
            rows=nrows,
            cols=ncols,
            subplot_titles=intervals,
            horizontal_spacing=0.08,
            vertical_spacing=0.18 if nrows > 1 else 0.10,
        )

        target_to_i = {target: idx for idx, target in enumerate(targets)}

        for interval_idx, interval_name in enumerate(intervals):
            row = (interval_idx // ncols) + 1
            col = (interval_idx % ncols) + 1
            sub = run_df[run_df['interval_name'].astype(str) == str(interval_name)].copy()
            sessions_present = set(sub['session_id'].astype(str))
            sessions = [session for session in all_sessions if session in sessions_present]
            if not sessions:
                continue
            session_tick_text = [session[:10] if _parse_session_date_token(session) else session for session in sessions]
            session_to_j = {session: idx for idx, session in enumerate(sessions)}
            separator_idx = next(
                (idx for idx, session in enumerate(sessions) if _parse_session_date_token(session) == separator_date),
                None,
            )

            z = np.full((len(targets), len(sessions)), np.nan, dtype=float)
            custom = np.full((len(targets), len(sessions), len(extra_cols)), np.nan, dtype=float)
            significance = np.zeros((len(targets), len(sessions)), dtype=bool) if significance_col else None
            for rec in sub.itertuples(index=False):
                i = target_to_i[str(rec.target_name)]
                j = session_to_j[str(rec.session_id)]
                value = getattr(rec, value_col)
                z[i, j] = float(value) if pd.notna(value) else np.nan
                for extra_idx, extra_col in enumerate(extra_cols):
                    raw = getattr(rec, extra_col)
                    custom[i, j, extra_idx] = float(raw) if pd.notna(raw) else np.nan
                if significance is not None:
                    significance[i, j] = _coerce_bool_flag(getattr(rec, significance_col))

            hover_lines = [
                f'{value_col}=%{{z:.3f}}',
            ]
            for extra_idx, extra_col in enumerate(extra_cols):
                hover_lines.append(f'{extra_col}=%{{customdata[{extra_idx}]:.3f}}')
            hovertemplate = 'interval=' + str(interval_name) + '<br>target=%{y}<br>session=%{x}<br>' + '<br>'.join(hover_lines) + '<extra></extra>'

            fig.add_trace(
                go.Heatmap(
                    x=sessions,
                    y=targets,
                    z=z,
                    customdata=custom,
                    coloraxis='coloraxis',
                    hovertemplate=hovertemplate,
                ),
                row=row,
                col=col,
            )

            if separator_idx is not None:
                axis_suffix = '' if interval_idx == 0 else str(interval_idx + 1)
                separator_x = separator_idx - 0.5
                fig.add_shape(
                    type='line',
                    x0=separator_x,
                    x1=separator_x,
                    y0=0,
                    y1=1,
                    xref=f'x{axis_suffix}',
                    yref=f'y{axis_suffix} domain',
                    line={'color': 'red', 'width': 2},
                )

            if significance is not None:
                marker_x = []
                marker_y = []
                for target_idx, target_name in enumerate(targets):
                    for session_idx, session_name in enumerate(sessions):
                        if significance[target_idx, session_idx] and np.isfinite(z[target_idx, session_idx]):
                            marker_x.append(session_name)
                            marker_y.append(target_name)
                if marker_x:
                    fig.add_trace(
                        go.Scatter(
                            x=marker_x,
                            y=marker_y,
                            mode='markers',
                            marker={
                                'size': significance_marker_size,
                                'color': 'rgba(255, 255, 255, 0.95)',
                                'line': {'color': 'black', 'width': 1.25},
                            },
                            name=significance_label,
                            legendgroup='shuffle_significance',
                            showlegend=interval_idx == 0,
                            hovertemplate=significance_label + '<br>target=%{y}<br>session=%{x}<extra></extra>',
                        ),
                        row=row,
                        col=col,
                    )
            fig.update_xaxes(
                tickmode='array',
                tickvals=sessions,
                ticktext=session_tick_text,
                tickangle=45,
                title_text='session',
                row=row,
                col=col,
            )
            fig.update_yaxes(title_text='target', row=row, col=col)

        coloraxis = {
            'colorscale': colorscale,
            'colorbar': {'title': value_col},
        }
        if zmin is not None:
            coloraxis['cmin'] = float(zmin)
        if zmax is not None:
            coloraxis['cmax'] = float(zmax)
        if zmid is not None:
            coloraxis['cmid'] = zmid

        fig.update_layout(
            template='plotly_white+pub',
            title=f'{title_prefix} | run={run_name}',
            coloraxis=coloraxis,
            height=max(200, int(230 * nrows)),
            width=max(720, int(250 * ncols)),
            margin={'l': 68, 'r': 36, 't': 60, 'b': 86},
        )
        fig.show()


### Session Progression Heatmaps

Each subplot is one interval. Columns are sessions with at least 10 trials for that interval, rows are decoded targets, and hover contains shuffle diagnostics plus trial and split counts. White marker dots indicate shuffle-significant cells after BH-FDR correction (`significant_shuffle_fdr_bh_0_05`).


In [ ]:
plot_session_interval_heatmaps(
    session_results_df,
    value_col='balanced_accuracy',
    title_prefix='Session-level balanced accuracy',
    extra_cols=['shuffle_null_mean', 'p_shuffle', 'p_shuffle_fdr_bh', 'n_trials', 'n_splits'],
    ncols=3,
    colorscale='Reds',
    zmid=None,
    zmin=0.5,
    zmax=0.9,
    significance_col='significant_shuffle_fdr_bh_0_05',
    significance_label='shuffle significant (BH-FDR q<0.05)',
    min_trials=10,
    significance_marker_size=4,
    interval_names=BALANCED_ACCURACY_INTERVALS,
    # excluded_targets=('Outcome',),
)


### Session-Level Shuffle Effect


In [ ]:
# plot_session_interval_heatmaps(
#     session_results_df,
#     value_col='shuffle_effect',
#     title_prefix='Session-level shuffle effect',
#     extra_cols=['shuffle_null_mean', 'p_shuffle', 'p_shuffle_fdr_bh', 'n_trials', 'n_splits'],
#     ncols=3,
#     colorscale='RdBu_r',
#     zmid=0.0,
# )


In [ ]:
def plot_session_feature_heatmaps(
    results_df: pd.DataFrame,
    value_col: str = 'balanced_accuracy',
    title_prefix: str = 'Session-level balanced accuracy by feature',
    extra_cols=None,
    ncols: int = 3,
    colorscale=None,
    zmid: float = 0.5,
    zmin=None,
    zmax=None,
    significance_col=None,
    significance_label: str = 'shuffle significant (p<0.05)',
    min_trials=None,
    significance_marker_size: float = 4.0,
    interval_names=None,
    excluded_targets=('Outcome',),
    row_height_px: int = 12,
    min_height_px: int = 300,
    panel_width_px: int = 190,
    x_tick_font_size: int = 8,
    y_tick_font_size: int = 8,
    shared_x_title: str = 'Feature',
    title_template=None,
    target_by_interval=None,
    min_width_px: int = 480,
    margin_l_px: int = 95,
    margin_r_px: int = 145,
    margin_t_px: int = 85,
    margin_b_px: int = 70,
    font_scale: float = 1.0,
    interval_label_map=None,
):
    if extra_cols is None:
        extra_cols = []

    if results_df.empty:
        print('No session-level results loaded.')
        return
    if value_col not in results_df.columns:
        print(f"Column '{value_col}' is not available in the loaded session results.")
        return

    extra_cols = [extra_col for extra_col in extra_cols if extra_col in results_df.columns]
    excluded_targets_lower = {str(target).lower() for target in excluded_targets}
    valid = results_df[results_df[value_col].notna()].copy()
    valid = valid[~valid['target_name'].astype(str).str.lower().isin(excluded_targets_lower)].copy()
    if min_trials is not None and 'n_trials' in valid.columns:
        valid = valid[pd.to_numeric(valid['n_trials'], errors='coerce') >= float(min_trials)].copy()
    target_by_interval = (
        {str(interval_name): str(target_name) for interval_name, target_name in target_by_interval.items()}
        if target_by_interval
        else None
    )
    if target_by_interval is not None:
        selected_targets = valid['interval_name'].astype(str).map(target_by_interval)
        valid = valid[selected_targets.eq(valid['target_name'].astype(str))].copy()
        if valid.empty:
            print('No rows remain after selecting interval-relevant targets.')
            return

    if valid.empty:
        trial_suffix = f' after filtering to n_trials >= {min_trials}' if min_trials is not None else ''
        print(f'No rows with {value_col}{trial_suffix} after excluding {sorted(excluded_targets)}.')
        return

    separator_date = datetime.strptime('2025-01-23', '%Y-%m-%d')

    def _scaled(size):
        return max(5, int(round(size * font_scale)))

    title_font_size = _scaled(13)
    subplot_title_font_size = _scaled(11)
    annotation_font_size = _scaled(11)
    colorbar_title_font_size = _scaled(10)
    colorbar_tick_font_size = _scaled(9)
    legend_font_size = _scaled(9)
    base_font_size = _scaled(10)

    for run_name, run_df in valid.groupby('run_name', sort=False):
        run_df, intervals = _filter_interval_rows(run_df, interval_names)
        if run_df.empty or not intervals:
            continue
        run_df = _add_shuffle_fdr_columns(run_df)

        features = _target_order(run_df['target_name'].astype(str))
        features = [feature for feature in features if feature.lower() not in excluded_targets_lower]
        features_by_interval = {}
        for interval_name in intervals:
            if target_by_interval is None:
                features_by_interval[str(interval_name)] = features
                continue
            selected_feature = target_by_interval.get(str(interval_name))
            interval_targets = set(run_df[run_df['interval_name'].astype(str) == str(interval_name)]['target_name'].astype(str))
            features_by_interval[str(interval_name)] = [selected_feature] if selected_feature in interval_targets else []

        all_sessions = sorted(run_df['session_id'].astype(str).unique().tolist(), key=_session_sort_key)
        assembly_match = re.search(r'(Assembly\d+)', str(run_name))
        assembly_label = assembly_match.group(1) if assembly_match else str(run_name)
        figure_title = (
            title_template.format(assembly=assembly_label, run_name=run_name, value_col=value_col)
            if title_template
            else f'{title_prefix} | run={run_name}'
        )

        subplot_titles = [
            interval_label_map.get(str(name), str(name)) if interval_label_map else str(name)
            for name in intervals
        ]
        nrows = int(np.ceil(len(intervals) / ncols))
        fig = make_subplots(
            rows=nrows,
            cols=ncols,
            subplot_titles=subplot_titles,
            horizontal_spacing=0.0,
            vertical_spacing=0.08 if nrows > 1 else 0.04,
        )

        for interval_idx, interval_name in enumerate(intervals):
            row = (interval_idx // ncols) + 1
            col = (interval_idx % ncols) + 1
            sub = run_df[run_df['interval_name'].astype(str) == str(interval_name)].copy()
            sessions = all_sessions
            interval_features = features_by_interval.get(str(interval_name), features)
            if not sessions or not interval_features:
                continue
            feature_to_j = {feature: idx for idx, feature in enumerate(interval_features)}

            session_tick_text = [session[:10] if _parse_session_date_token(session) else session for session in sessions]
            session_to_i = {session: idx for idx, session in enumerate(sessions)}
            separator_idx = next(
                (idx for idx, session in enumerate(sessions) if _parse_session_date_token(session) == separator_date),
                None,
            )

            z = np.full((len(sessions), len(interval_features)), np.nan, dtype=float)
            custom = np.full((len(sessions), len(interval_features), len(extra_cols)), np.nan, dtype=float)
            significance = np.zeros((len(sessions), len(interval_features)), dtype=bool) if significance_col else None
            for rec in sub.itertuples(index=False):
                target_name = str(rec.target_name)
                if target_name not in feature_to_j:
                    continue
                i = session_to_i[str(rec.session_id)]
                j = feature_to_j[target_name]
                value = getattr(rec, value_col)
                z[i, j] = float(value) if pd.notna(value) else np.nan
                for extra_idx, extra_col in enumerate(extra_cols):
                    raw = getattr(rec, extra_col)
                    custom[i, j, extra_idx] = float(raw) if pd.notna(raw) else np.nan
                if significance is not None:
                    significance[i, j] = _coerce_bool_flag(getattr(rec, significance_col))

            hover_lines = [f'{value_col}=%{{z:.3f}}']
            for extra_idx, extra_col in enumerate(extra_cols):
                hover_lines.append(f'{extra_col}=%{{customdata[{extra_idx}]:.3f}}')
            hovertemplate = 'interval=' + str(interval_name) + '<br>feature=%{x}<br>session=%{y}<br>' + '<br>'.join(hover_lines) + '<extra></extra>'

            fig.add_trace(
                go.Heatmap(
                    x=interval_features,
                    y=sessions,
                    z=z,
                    customdata=custom,
                    coloraxis='coloraxis',
                    xgap=0,
                    ygap=0,
                    hovertemplate=hovertemplate,
                ),
                row=row,
                col=col,
            )

            if separator_idx is not None:
                axis_suffix = '' if interval_idx == 0 else str(interval_idx + 1)
                separator_y = separator_idx - 0.5
                fig.add_shape(
                    type='line',
                    x0=0,
                    x1=1,
                    y0=separator_y,
                    y1=separator_y,
                    xref=f'x{axis_suffix} domain',
                    yref=f'y{axis_suffix}',
                    line={'color': 'red', 'width': 2},
                )

            if significance is not None:
                marker_x = []
                marker_y = []
                for session_idx, session_name in enumerate(sessions):
                    for feature_idx, feature_name in enumerate(interval_features):
                        if significance[session_idx, feature_idx] and np.isfinite(z[session_idx, feature_idx]):
                            marker_x.append(feature_name)
                            marker_y.append(session_name)
                if marker_x:
                    fig.add_trace(
                        go.Scatter(
                            x=marker_x,
                            y=marker_y,
                            mode='markers',
                            marker={
                                'size': significance_marker_size,
                                'color': 'rgba(255, 255, 255, 0.95)',
                                'line': {'color': 'black', 'width': 1.25},
                            },
                            name=significance_label,
                            legendgroup='shuffle_significance',
                            showlegend=interval_idx == 0,
                            hovertemplate=significance_label + '<br>feature=%{x}<br>session=%{y}<extra></extra>',
                        ),
                        row=row,
                        col=col,
                    )

            fig.update_xaxes(
                title_text=None,
                tickangle=0,
                tickfont={'size': x_tick_font_size},
                ticks='outside',
                showline=True,
                linewidth=1,
                linecolor='black',
                mirror=True,
                row=row,
                col=col,
            )
            fig.update_yaxes(
                tickmode='array',
                tickvals=sessions,
                ticktext=session_tick_text,
                title_text='session' if col == 1 else None,
                showticklabels=col == 1,
                ticks='outside' if col == 1 else '',
                tickfont={'size': y_tick_font_size},
                showline=True,
                linewidth=1,
                linecolor='black',
                mirror=True,
                autorange='reversed',
                row=row,
                col=col,
            )

        for row_idx in range(nrows):
            for boundary_col in range(1, ncols):
                left_interval_idx = row_idx * ncols + boundary_col - 1
                right_interval_idx = left_interval_idx + 1
                if right_interval_idx >= len(intervals):
                    continue
                axis_suffix = '' if left_interval_idx == 0 else str(left_interval_idx + 1)
                xaxis = fig.layout[f'xaxis{axis_suffix}']
                yaxis = fig.layout[f'yaxis{axis_suffix}']
                fig.add_shape(
                    type='line',
                    x0=xaxis.domain[1],
                    x1=xaxis.domain[1],
                    y0=yaxis.domain[0],
                    y1=yaxis.domain[1],
                    xref='paper',
                    yref='paper',
                    line={'color': 'black', 'width': 2},
                )

        coloraxis = {
            'colorscale': colorscale,
            'colorbar': {
                'title': {'text': value_col, 'font': {'size': colorbar_title_font_size}},
                'tickfont': {'size': colorbar_tick_font_size},
                'thickness': 12,
                'len': 0.58,
                'x': 1.035,
                'y': 0.40,
            },
        }
        if zmin is not None:
            coloraxis['cmin'] = float(zmin)
        if zmax is not None:
            coloraxis['cmax'] = float(zmax)
        if zmid is not None:
            coloraxis['cmid'] = zmid

        if shared_x_title:
            fig.add_annotation(
                text=shared_x_title,
                x=0.5,
                y=-0.16,
                xref='paper',
                yref='paper',
                showarrow=False,
                font={'size': annotation_font_size, 'color': 'black'},
            )

        for annotation in fig.layout.annotations:
            current_size = annotation.font.size if (annotation.font and annotation.font.size) else subplot_title_font_size
            annotation.font = {'size': current_size, 'color': 'black'}

        fig.update_layout(
            template='plotly_white+pub',
            title={
                'text': figure_title,
                'x': 0.02,
                'xanchor': 'left',
                'font': {'size': title_font_size, 'color': 'black'},
            },
            font={'family': 'Arial, Helvetica, sans-serif', 'size': base_font_size, 'color': 'black'},
            coloraxis=coloraxis,
            legend={
                'orientation': 'v',
                'x': 1.035,
                'xanchor': 'left',
                'y': 0.86,
                'yanchor': 'bottom',
                'bgcolor': 'rgba(255,255,255,0)',
                'borderwidth': 0,
                'font': {'size': legend_font_size},
                'itemsizing': 'constant',
            },
            height=max(int(min_height_px), int(row_height_px * len(all_sessions) * nrows + 140)),
            width=max(int(min_width_px), int(panel_width_px * min(ncols, len(intervals)) + margin_l_px + margin_r_px)),
            margin={'l': margin_l_px, 'r': margin_r_px, 't': margin_t_px, 'b': margin_b_px},
            plot_bgcolor='white',
            paper_bgcolor='white',
        )
        fig.show()


plot_session_feature_heatmaps(
    session_results_df,
    value_col='balanced_accuracy',
    title_prefix='Session-level balanced accuracy by feature',
    extra_cols=['shuffle_null_mean', 'p_shuffle', 'p_shuffle_fdr_bh', 'n_trials', 'n_splits'],
    ncols=3,
    colorscale='Reds',
    zmid=None,
    zmin=0.5,
    zmax=0.9,
    significance_col='significant_shuffle_fdr_bh_0_05',
    significance_label='shuffle significant (BH-FDR q<0.05)',
    min_trials=10,
    significance_marker_size=4,
    interval_names=BALANCED_ACCURACY_INTERVALS,
    excluded_targets=('Outcome',),
    row_height_px=12,
    min_height_px=300,
    panel_width_px=120,
    x_tick_font_size=8,
    y_tick_font_size=8,
    shared_x_title='Feature',
    title_template='{assembly} - sessionwise balanced accuracy by feature',
)


In [ ]:
# Heatmap for only the most relevant feature of the interval and adjusted BH-FDR test
RELEVANT_TARGET_BY_INTERVAL = {
    'cue_entry_interval': 'Cue',
    'R1_entry_interval': 'R1 choice',
    'R2_entry_interval': 'R2 choice',
}
INTERVAL_LABEL_MAP = {
    'cue_entry_interval': 'Cue entry',
    'R1_entry_interval': 'R1 entry',
    'R2_entry_interval': 'R2 entry',
}

plot_session_feature_heatmaps(
    session_results_df,
    value_col='balanced_accuracy',
    title_prefix='',
    extra_cols=['shuffle_null_mean', 'p_shuffle', 'p_shuffle_fdr_bh', 'n_trials', 'n_splits'],
    ncols=3,
    colorscale='Reds',
    zmid=None,
    zmin=0.5,
    zmax=0.9,
    significance_col='significant_shuffle_fdr_bh_0_05',
    significance_label='BH-FDR q<0.05',
    min_trials=10,
    significance_marker_size=2.5,
    interval_names=BALANCED_ACCURACY_INTERVALS,
    excluded_targets=('Outcome',),
    row_height_px=10,
    min_height_px=330,
    panel_width_px=58,
    x_tick_font_size=7,
    y_tick_font_size=7,
    shared_x_title='',
    title_template='{assembly}',
    target_by_interval=RELEVANT_TARGET_BY_INTERVAL,
    interval_label_map=INTERVAL_LABEL_MAP,
    min_width_px=420,
    margin_l_px=70,
    margin_r_px=95,
    margin_t_px=55,
    margin_b_px=45,
    font_scale=0.78,
)


In [ ]:
# Minimal helpers for the focused cue x choice accuracy matrix.
import sys


def _short_session_id(session_name):
    return '_'.join(str(session_name).split('_')[:2])


def _ensure_session_decoder_imports():
    root = next(
        p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (p / 'analytics_processing').is_dir()
    )
    analysis_dir = root / 'scripted_plotting' / 'animal_6_analysis'
    for p in [analysis_dir, root, root.parent]:
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))

    from baseVR.base_functionality import init_import_paths
    init_import_paths()
    from CustomLogger import CustomLogger as Logger
    from analytics_processing import analytics
    from interval_decoding_pipeline import prepare_interval_table
    from interval_session_decoding_pipeline import (
        SessionDecodeConfig,
        _build_cv_splits,
        _build_session_trial_matrix,
        _fit_predict,
    )
    Logger().init_logger(None, None, logging_level='WARNING')
    return analytics, prepare_interval_table, SessionDecodeConfig, _build_session_trial_matrix, _build_cv_splits, _fit_predict


def _resolve_full_session_names(short_session_ids, animal_ids=(6,), paradigm_ids=(1100,)):
    from analytics_processing.sessions_from_nas_parsing import fullfnames2snames, sessionlist_fullfnames_from_args

    excluded = [
        '2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min',
        '2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min',
        '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min',
    ]
    session_dirs = sessionlist_fullfnames_from_args(list(paradigm_ids), list(animal_ids), None, excl_session_names=excluded)[0]
    full_names = fullfnames2snames(session_dirs)
    lookup = {_short_session_id(name): name for name in full_names}
    return [name if len(str(name).split('_')) >= 4 else lookup[str(name)] for name in short_session_ids]


def _session_cfg(row, SessionDecodeConfig):
    cfg = {}
    if 'session_bundle' in globals() and row.get('run_name') in session_bundle.get('config_df', pd.DataFrame()).index:
        cfg = session_bundle['config_df'].loc[row['run_name']].dropna().to_dict()
    cfg = {key: value for key, value in cfg.items() if key in SessionDecodeConfig.__dataclass_fields__}
    cfg['assembly_col'] = str(row['assembly'])
    cfg['min_session_date'] = None
    return SessionDecodeConfig(**cfg)


def _class_labels(target_col, raw_values):
    labels = {
        'cue': {1: 'cue 1', 2: 'cue 2'},
        'choice_R1': {0: 'not R1', 1: 'R1'},
        'choice_R2': {0: 'not R2', 1: 'R2'},
        'outcome_binary': {0: 'incorrect', 1: 'correct'},
    }.get(str(target_col), {})
    return [labels.get(int(value), f'{target_col}={int(value)}') for value in raw_values]


def _compute_confusion_for_row(row, ens_df, helpers, cache):
    prepare_interval_table, SessionDecodeConfig, build_trial_matrix, build_cv_splits, fit_predict = helpers
    key = (str(row['assembly']), str(row['target_col']))
    if key not in cache:
        cfg = _session_cfg(row, SessionDecodeConfig)
        df, session_col, trial_col, interval_col, feature_cols, _, _ = prepare_interval_table(ens_df, cfg)
        matrix = build_trial_matrix(df, session_col, trial_col, interval_col, feature_cols[0], str(row['target_col']))
        cache[key] = matrix, cfg, session_col, trial_col, interval_col, str(row['target_col'])

    matrix, cfg, session_col, trial_col, interval_col, target_col = cache[key]
    session_ids = matrix[session_col].astype(str)
    selected = matrix[
        ((session_ids == str(row['session_id'])) | (session_ids.map(_short_session_id) == str(row['session_id'])))
        & matrix[interval_col].astype(str).eq(str(row['interval_name']))
    ].sort_values(trial_col)

    feature_cols = [c for c in selected.columns if c not in {session_col, trial_col, interval_col, target_col}]
    feature_cols = [c for c in feature_cols if selected[c].notna().any()]
    y_raw = selected[target_col].to_numpy(dtype=int)
    raw_values = np.sort(np.unique(y_raw))
    y = np.where(y_raw == raw_values[0], 0, 1).astype(np.int64)
    x = selected[feature_cols].to_numpy(dtype=np.float32)
    splits, _ = build_cv_splits(y, cfg)

    y_pred = np.full(y.shape, -1, dtype=np.int64)
    for fold_idx, (train_idx, test_idx) in enumerate(splits):
        y_pred[test_idx] = fit_predict(x[train_idx], y[train_idx], x[test_idx], int(cfg.random_state + fold_idx))

    labels = _class_labels(target_col, raw_values)
    return {
        'predictions': pd.DataFrame({
            'trial_id': selected[trial_col].to_numpy(),
            'true_class': [labels[i] for i in y],
            'predicted_class': [labels[i] for i in y_pred],
        })
    }


### Decoder accuracy by true cue x choice

A focused red matrix for one selected session and interval. Rows are true cue/choice combinations; columns are the cue and choice decoders.


In [ ]:
CONJUGATE_SESSION_ID = '2024-12-10_17-20'
CONJUGATE_INTERVAL = 'R1_entry_interval'
CONJUGATE_TARGETS = ['Cue', 'R1 choice']
CONJUGATE_ASSEMBLY = None  # None -> use the assembly with the best mean bacc across both targets.

rows = session_results_df.copy()
rows['assembly'] = rows['assembly'] if 'assembly' in rows else rows['run_name'].astype(str).str.extract(r'(Assembly\d+)', expand=False)
rows = rows[
    rows['session_id'].astype(str).str.startswith(CONJUGATE_SESSION_ID)
    & rows['interval_name'].astype(str).eq(CONJUGATE_INTERVAL)
    & rows['target_name'].astype(str).isin(CONJUGATE_TARGETS)
].copy()
rows['balanced_accuracy'] = pd.to_numeric(rows['balanced_accuracy'], errors='coerce')
assembly = CONJUGATE_ASSEMBLY or rows.groupby('assembly')['balanced_accuracy'].mean().idxmax()
rows = rows[rows['assembly'].astype(str).eq(str(assembly))]

analytics_mod, prepare_interval_table, SessionDecodeConfig, build_trial_matrix, build_cv_splits, fit_predict = _ensure_session_decoder_imports()
ens_df = analytics_mod.get_analytics('EnsembleT0Projection', session_names=_resolve_full_session_names([CONJUGATE_SESSION_ID]))
helpers = (prepare_interval_table, SessionDecodeConfig, build_trial_matrix, build_cv_splits, fit_predict)
cache = {}
items = {
    target: _compute_confusion_for_row(rows[rows['target_name'].astype(str).eq(target)].iloc[0].to_dict(), ens_df, helpers, cache)
    for target in CONJUGATE_TARGETS
}

cue = items['Cue']['predictions'][['trial_id', 'true_class', 'predicted_class']].rename(columns={'true_class': 'cue_true', 'predicted_class': 'cue_pred'})
choice = items['R1 choice']['predictions'][['trial_id', 'true_class', 'predicted_class']].rename(columns={'true_class': 'choice_true', 'predicted_class': 'choice_pred'})
pred = cue.merge(choice, on='trial_id')
pred['Cue decoder'] = pred['cue_true'].eq(pred['cue_pred'])
pred['R1 choice decoder'] = pred['choice_true'].eq(pred['choice_pred'])
pred['true cue x choice'] = pred['cue_true'] + ' | ' + pred['choice_true']

group_order = [
    f'{cue_label} | {choice_label}'
    for cue_label in ['cue 1', 'cue 2']
    for choice_label in ['not R1', 'R1']
    if ((pred['cue_true'] == cue_label) & (pred['choice_true'] == choice_label)).any()
]
decoder_cols = ['Cue decoder', 'R1 choice decoder']
accuracy = pred.groupby('true cue x choice')[decoder_cols].mean().reindex(group_order)
counts = pred.groupby('true cue x choice').size().reindex(group_order)
text = [[f'{accuracy.loc[group, decoder]:.2f}<br>n={int(counts.loc[group])}' for decoder in decoder_cols] for group in group_order]

fig = go.Figure(go.Heatmap(
    z=accuracy.to_numpy(dtype=float),
    x=decoder_cols,
    y=group_order,
    text=text,
    texttemplate='%{text}',
    colorscale='Reds',
    zmin=0.5,
    zmax=1.0,
    colorbar={'title': {'text': 'Accuracy', 'font': {'size': 9}}, 'thickness': 10, 'len': 0.7, 'tickfont': {'size': 8}},
    hovertemplate='true cue x choice=%{y}<br>decoder=%{x}<br>accuracy=%{z:.3f}<extra></extra>',
))
fig.update_yaxes(autorange='reversed', title_text='true cue x choice')
fig.update_xaxes(title_text='decoder')
fig.update_layout(
    template='plotly_white+pub',
    title={'text': f'{assembly} | cue x choice decoder accuracy', 'x': 0.02, 'xanchor': 'left', 'font': {'color': 'black', 'size': 11}},
    width=430,
    height=max(240, 60 + 42 * len(group_order)),
    margin={'l': 120, 'r': 56, 't': 54, 'b': 46},
    font={'family': 'Arial, Helvetica, sans-serif', 'size': 9, 'color': 'black'},
)
fig.show()


### Across-session primary decoding-effect test

This plot treats sessions as replicates. For each assembly, decoded target, and interval, it tests whether the sessionwise shuffle effect (`balanced_accuracy - shuffle_null_mean`) is consistently above zero using a one-sided sign-flip test across sessions. Error bars are estimated with a bootstrap across sessions, and p-values are BH-FDR corrected across the shown target x interval tests within each assembly.


In [ ]:
import hashlib

SESSION_EFFECT_INTERVALS = BALANCED_ACCURACY_INTERVALS
SESSION_EFFECT_EXCLUDED_TARGETS = ('Outcome',)
SESSION_EFFECT_MIN_TRIALS = 10
SESSION_EFFECT_MIN_SESSIONS = 5
SESSION_EFFECT_SIGNFLIP_N = 50_000
SESSION_EFFECT_BOOTSTRAP_N = 10_000
SESSION_EFFECT_RANDOM_STATE = 42
SESSION_EFFECT_FDR_SCOPE = 'per_assembly'  # 'per_assembly' or 'global'


def _stable_int_seed(*parts, base_seed: int = SESSION_EFFECT_RANDOM_STATE) -> int:
    payload = '||'.join([str(base_seed), *[str(part) for part in parts]]).encode('utf-8')
    return int(hashlib.md5(payload).hexdigest()[:8], 16)


def _one_sided_signflip_pvalue(effects, n_randomizations: int, seed: int) -> tuple[float, float]:
    effects = np.asarray(effects, dtype=float)
    effects = effects[np.isfinite(effects)]
    if effects.size == 0:
        return np.nan, np.nan

    observed = float(np.mean(effects))
    rng = np.random.default_rng(int(seed))
    signs = rng.choice(np.array([-1.0, 1.0]), size=(int(n_randomizations), effects.size), replace=True)
    null_stats = (signs * effects).mean(axis=1)
    p_value = float((1.0 + np.sum(null_stats >= observed)) / (1.0 + null_stats.size))
    return observed, p_value


def _bootstrap_mean_ci(effects, n_bootstrap: int, seed: int, ci: float = 0.95) -> tuple[float, float]:
    effects = np.asarray(effects, dtype=float)
    effects = effects[np.isfinite(effects)]
    if effects.size == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(int(seed))
    sample_idx = rng.integers(0, effects.size, size=(int(n_bootstrap), effects.size), endpoint=False)
    boot_means = effects[sample_idx].mean(axis=1)
    low_q = (1.0 - float(ci)) / 2.0
    high_q = 1.0 - low_q
    return float(np.quantile(boot_means, low_q)), float(np.quantile(boot_means, high_q))


def _prepare_session_effect_rows(results_df: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    out = results_df.copy()
    if 'assembly' not in out.columns and 'run_name' in out.columns:
        out['assembly'] = out['run_name'].astype(str).str.extract(r'(Assembly\d+)', expand=False)

    if SESSION_EFFECT_INTERVALS is not None:
        out = out[out['interval_name'].astype(str).isin([str(value) for value in SESSION_EFFECT_INTERVALS])].copy()
    excluded = {str(target).lower() for target in SESSION_EFFECT_EXCLUDED_TARGETS}
    out = out[~out['target_name'].astype(str).str.lower().isin(excluded)].copy()

    if SESSION_EFFECT_MIN_TRIALS is not None and 'n_trials' in out.columns:
        out = out[pd.to_numeric(out['n_trials'], errors='coerce') >= float(SESSION_EFFECT_MIN_TRIALS)].copy()

    out['balanced_accuracy_numeric'] = pd.to_numeric(out['balanced_accuracy'], errors='coerce')
    out['shuffle_null_mean_numeric'] = pd.to_numeric(out['shuffle_null_mean'], errors='coerce')
    out['session_shuffle_effect'] = out['balanced_accuracy_numeric'] - out['shuffle_null_mean_numeric']
    out = out.dropna(subset=['assembly', 'session_id', 'target_name', 'interval_name', 'session_shuffle_effect']).copy()
    return out


def compute_across_session_effect_summary(results_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    effect_rows = _prepare_session_effect_rows(results_df)
    if effect_rows.empty:
        return pd.DataFrame(), effect_rows

    summary_rows = []
    group_cols = ['assembly', 'run_name', 'target_name', 'interval_name']
    for group_key, group_df in effect_rows.groupby(group_cols, sort=False):
        group_dict = dict(zip(group_cols, group_key if isinstance(group_key, tuple) else (group_key,)))
        session_effects = (
            group_df.groupby('session_id', sort=False)['session_shuffle_effect']
            .mean()
            .dropna()
            .to_numpy(dtype=float)
        )
        n_sessions = int(session_effects.size)
        if n_sessions < int(SESSION_EFFECT_MIN_SESSIONS):
            continue

        seed_parts = [group_dict.get('assembly'), group_dict.get('target_name'), group_dict.get('interval_name')]
        observed_mean, p_signflip = _one_sided_signflip_pvalue(
            session_effects,
            n_randomizations=int(SESSION_EFFECT_SIGNFLIP_N),
            seed=_stable_int_seed(*seed_parts, 'signflip'),
        )
        ci_low, ci_high = _bootstrap_mean_ci(
            session_effects,
            n_bootstrap=int(SESSION_EFFECT_BOOTSTRAP_N),
            seed=_stable_int_seed(*seed_parts, 'bootstrap'),
        )
        summary_rows.append({
            **group_dict,
            'n_sessions': n_sessions,
            'mean_shuffle_effect': observed_mean,
            'median_shuffle_effect': float(np.median(session_effects)),
            'bootstrap_ci_low': ci_low,
            'bootstrap_ci_high': ci_high,
            'p_signflip_mean_gt0': p_signflip,
        })

    summary_df = pd.DataFrame(summary_rows)
    if summary_df.empty:
        return summary_df, effect_rows

    summary_df['q_signflip_fdr_bh'] = np.nan
    if SESSION_EFFECT_FDR_SCOPE == 'global':
        summary_df['q_signflip_fdr_bh'] = _bh_fdr(summary_df['p_signflip_mean_gt0'].to_numpy(dtype=float))
    elif SESSION_EFFECT_FDR_SCOPE == 'per_assembly':
        for assembly, idx in summary_df.groupby('assembly').groups.items():
            idx = list(idx)
            summary_df.loc[idx, 'q_signflip_fdr_bh'] = _bh_fdr(summary_df.loc[idx, 'p_signflip_mean_gt0'].to_numpy(dtype=float))
    else:
        raise ValueError("SESSION_EFFECT_FDR_SCOPE must be 'per_assembly' or 'global'.")
    summary_df['significant_fdr_bh_0_05'] = summary_df['q_signflip_fdr_bh'] < 0.05
    return summary_df, effect_rows


def _short_interval_label(interval_name: str) -> str:
    return str(interval_name).replace('_interval', '').replace('_', ' ')




def _session_effect_test_order(summary_df: pd.DataFrame) -> list[tuple[str, str]]:
    excluded_lower = {str(target).lower() for target in SESSION_EFFECT_EXCLUDED_TARGETS}
    present_targets = set(summary_df['target_name'].astype(str))
    target_order = [
        target for target in TARGET_ORDER
        if target.lower() not in excluded_lower and target in present_targets
    ]
    target_order += sorted(present_targets - set(target_order))
    interval_order = _resolve_interval_order(summary_df['interval_name'], SESSION_EFFECT_INTERVALS)
    return [
        (target, interval_name)
        for interval_name in interval_order
        for target in target_order
        if ((summary_df['target_name'].astype(str) == target) & (summary_df['interval_name'].astype(str) == interval_name)).any()
    ]


def plot_across_session_effect_summary(summary_df: pd.DataFrame) -> None:
    if summary_df.empty:
        print('No across-session effect summary rows to plot.')
        return

    x_order = _session_effect_test_order(summary_df)
    x_labels = [f'{target}<br>{_short_interval_label(interval_name)}' for target, interval_name in x_order]
    assemblies = sorted(summary_df['assembly'].astype(str).unique().tolist(), key=_assembly_sort_key)

    z = np.full((len(assemblies), len(x_order)), np.nan, dtype=float)
    text = np.full((len(assemblies), len(x_order)), '', dtype=object)
    custom = np.full((len(assemblies), len(x_order), 6), np.nan, dtype=float)
    for rec in summary_df.itertuples(index=False):
        key = (str(rec.target_name), str(rec.interval_name))
        if key not in x_order:
            continue
        i = assemblies.index(str(rec.assembly))
        j = x_order.index(key)
        z[i, j] = float(rec.mean_shuffle_effect)
        text[i, j] = f'{rec.mean_shuffle_effect:.3f}<br>q={rec.q_signflip_fdr_bh:.3g}'
        custom[i, j, :] = [
            float(rec.n_sessions),
            float(rec.median_shuffle_effect),
            float(rec.bootstrap_ci_low),
            float(rec.bootstrap_ci_high),
            float(rec.p_signflip_mean_gt0),
            float(rec.q_signflip_fdr_bh),
        ]

    max_abs = np.nanmax(np.abs(z)) if np.isfinite(z).any() else 0.1
    max_abs = max(float(max_abs), 0.02)
    fig = go.Figure()
    fig.add_trace(go.Heatmap(
        z=z,
        x=x_labels,
        y=assemblies,
        text=text,
        texttemplate='%{text}',
        customdata=custom,
        colorscale='RdBu_r',
        zmin=-max_abs,
        zmax=max_abs,
        colorbar={'title': {'text': 'Mean shuffle effect', 'font': {'size': 9}}, 'thickness': 10, 'len': 0.7, 'tickfont': {'size': 8}},
        hovertemplate=(
            'assembly=%{y}<br>test=%{x}<br>'
            'mean effect=%{z:.4f}<br>'
            'n sessions=%{customdata[0]:.0f}<br>'
            'median effect=%{customdata[1]:.4f}<br>'
            'bootstrap CI=[%{customdata[2]:.4f}, %{customdata[3]:.4f}]<br>'
            'sign-flip p=%{customdata[4]:.4g}<br>'
            'BH-FDR q=%{customdata[5]:.4g}<extra></extra>'
        ),
    ))

    sig_df = summary_df[summary_df['significant_fdr_bh_0_05']].copy()
    if not sig_df.empty:
        sig_df['x_label'] = [
            f'{target}<br>{_short_interval_label(interval_name)}'
            for target, interval_name in zip(sig_df['target_name'], sig_df['interval_name'])
        ]
        fig.add_trace(go.Scatter(
            x=sig_df['x_label'],
            y=sig_df['assembly'].astype(str),
            mode='markers',
            marker={'symbol': 'circle-open', 'size': 9, 'color': 'black', 'line': {'width': 1.5}},
            name='BH-FDR q<0.05',
            hovertemplate='assembly=%{y}<br>test=%{x}<br>BH-FDR q<0.05<extra></extra>',
        ))

    fig.update_layout(
        template='plotly_white+pub',
        title={
            'text': f'Across-session decoding effect: one-sided sign-flip test ({SESSION_EFFECT_FDR_SCOPE} BH-FDR)',
            'x': 0.02,
            'xanchor': 'left',
            'font': {'size': 12, 'color': 'black'},
        },
        font={'family': 'Arial, Helvetica, sans-serif', 'size': 9, 'color': 'black'},
        width=max(540, 60 * len(x_labels) + 175),
        height=max(340, 22 * len(assemblies) + 120),
        margin={'l': 85, 'r': 118, 't': 70, 'b': 78},
        paper_bgcolor='white',
        plot_bgcolor='white',
    )
    fig.update_xaxes(title_text='decoded target x interval')
    fig.update_yaxes(title_text='assembly', autorange='reversed')
    fig.show()


session_effect_summary_df, session_effect_rows_df = compute_across_session_effect_summary(session_results_df)
if session_effect_summary_df.empty:
    print('No across-session primary-test rows were available after filtering.')
else:
    display(
        session_effect_summary_df.sort_values(['q_signflip_fdr_bh', 'p_signflip_mean_gt0'])[
            [
                'assembly', 'target_name', 'interval_name', 'n_sessions',
                'mean_shuffle_effect', 'median_shuffle_effect', 'bootstrap_ci_low', 'bootstrap_ci_high',
                'p_signflip_mean_gt0', 'q_signflip_fdr_bh', 'significant_fdr_bh_0_05'
            ]
        ].head(30)
    )
    plot_across_session_effect_summary(session_effect_summary_df)


In [ ]:
# Joint cross-feature encoding check for highlighted ensembles.
import math

JOINT_FEATURE_ASSEMBLIES = ['Assembly007', 'Assembly023']
JOINT_FEATURE_TARGETS = ['Cue', 'R1 choice', 'R2 choice']
JOINT_FEATURE_INTERVALS = BALANCED_ACCURACY_INTERVALS
JOINT_FEATURE_MIN_TRIALS = 10


def _fisher_exact_greater(a: int, b: int, c: int, d: int) -> float:
    total = int(a + b + c + d)
    if total == 0:
        return np.nan
    row1 = int(a + b)
    col1 = int(a + c)
    denominator = math.comb(total, row1)
    max_successes = min(row1, col1)
    p_value = 0.0
    for successes in range(int(a), max_successes + 1):
        p_value += math.comb(col1, successes) * math.comb(total - col1, row1 - successes) / denominator
    return float(min(p_value, 1.0))


def _joint_interval_label(interval_name: str) -> str:
    fallback_label = str(interval_name).replace('_interval', '').replace('_', ' ')
    return INTERVAL_LABEL_MAP.get(str(interval_name), fallback_label)


def _joint_feature_label(target_name: str, interval_name: str) -> str:
    return f'{target_name}<br>{_joint_interval_label(interval_name)}'


def prepare_joint_cross_feature_rows(results_df: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    out = results_df.copy()
    if 'assembly' not in out.columns and 'run_name' in out.columns:
        out['assembly'] = out['run_name'].astype(str).str.extract(r'(Assembly\d+)', expand=False)

    out = out[out['assembly'].astype(str).isin(JOINT_FEATURE_ASSEMBLIES)].copy()
    out = out[out['interval_name'].astype(str).isin([str(value) for value in JOINT_FEATURE_INTERVALS])].copy()
    out = out[out['target_name'].astype(str).isin(JOINT_FEATURE_TARGETS)].copy()
    out = out[pd.to_numeric(out['n_trials'], errors='coerce') >= float(JOINT_FEATURE_MIN_TRIALS)].copy()
    if out.empty:
        return out

    for column in ['balanced_accuracy', 'shuffle_null_mean', 'shuffle_effect', 'p_shuffle']:
        if column in out.columns:
            out[column] = pd.to_numeric(out[column], errors='coerce')
    out['shuffle_effect'] = out['balanced_accuracy'] - out['shuffle_null_mean']
    out['is_primary_pair'] = out['interval_name'].astype(str).map(RELEVANT_TARGET_BY_INTERVAL).eq(out['target_name'].astype(str))
    out['primary_q_fdr_bh'] = np.nan
    out['feature_matrix_q_fdr_bh'] = np.nan

    for assembly, assembly_idx in out.groupby('assembly').groups.items():
        assembly_idx = list(assembly_idx)
        primary_idx = out.loc[assembly_idx].index[out.loc[assembly_idx, 'is_primary_pair']].tolist()
        out.loc[assembly_idx, 'feature_matrix_q_fdr_bh'] = _bh_fdr(out.loc[assembly_idx, 'p_shuffle'].to_numpy(dtype=float))
        out.loc[primary_idx, 'primary_q_fdr_bh'] = _bh_fdr(out.loc[primary_idx, 'p_shuffle'].to_numpy(dtype=float))

    out['primary_significant'] = out['primary_q_fdr_bh'] < 0.05
    out['feature_matrix_significant'] = out['feature_matrix_q_fdr_bh'] < 0.05
    out['non_relevant_significant'] = (~out['is_primary_pair']) & out['feature_matrix_significant']
    out['decoded_feature_interval'] = [
        _joint_feature_label(target_name, interval_name)
        for target_name, interval_name in zip(out['target_name'], out['interval_name'])
    ]
    return out


def compute_joint_cross_feature_tables(joint_rows: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if joint_rows.empty:
        return pd.DataFrame(), pd.DataFrame()

    cross_pairs = [
        (target_name, interval_name)
        for interval_name in JOINT_FEATURE_INTERVALS
        for target_name in JOINT_FEATURE_TARGETS
        if RELEVANT_TARGET_BY_INTERVAL.get(str(interval_name)) != str(target_name)
    ]
    primary_pairs = [(interval_name, RELEVANT_TARGET_BY_INTERVAL[interval_name]) for interval_name in JOINT_FEATURE_INTERVALS]
    overlap_rows = []
    joint_session_rows = []

    for assembly, assembly_df in joint_rows.groupby('assembly', sort=False):
        available_sessions = sorted(assembly_df['session_id'].astype(str).unique().tolist(), key=_session_sort_key)
        by_key = {
            (str(rec.session_id), str(rec.target_name), str(rec.interval_name)): rec
            for rec in assembly_df.itertuples(index=False)
        }
        for primary_interval, primary_target in primary_pairs:
            for query_target, query_interval in cross_pairs:
                eligible_sessions = [
                    session_id for session_id in available_sessions
                    if (session_id, primary_target, primary_interval) in by_key
                    and (session_id, query_target, query_interval) in by_key
                ]
                primary_sig = {
                    session_id for session_id in eligible_sessions
                    if bool(getattr(by_key[(session_id, primary_target, primary_interval)], 'primary_significant'))
                }
                query_sig = {
                    session_id for session_id in eligible_sessions
                    if bool(getattr(by_key[(session_id, query_target, query_interval)], 'non_relevant_significant'))
                }
                joint_sessions = sorted(primary_sig & query_sig, key=_session_sort_key)
                a = len(joint_sessions)
                b = len(primary_sig - query_sig)
                c = len(query_sig - primary_sig)
                d = len(set(eligible_sessions) - primary_sig - query_sig)
                p_value = _fisher_exact_greater(a, b, c, d) if primary_sig and query_sig else np.nan

                overlap_rows.append({
                    'assembly': assembly,
                    'primary_feature': primary_target,
                    'primary_interval': primary_interval,
                    'cross_feature': query_target,
                    'cross_interval': query_interval,
                    'n_eligible_sessions': len(eligible_sessions),
                    'n_primary_significant': len(primary_sig),
                    'n_cross_significant': len(query_sig),
                    'n_joint_sessions': a,
                    'p_fisher_greater': p_value,
                    'joint_sessions': ', '.join(joint_sessions),
                })

                for session_id in joint_sessions:
                    primary_rec = by_key[(session_id, primary_target, primary_interval)]
                    cross_rec = by_key[(session_id, query_target, query_interval)]
                    joint_session_rows.append({
                        'assembly': assembly,
                        'session_id': session_id,
                        'primary_feature': primary_target,
                        'primary_interval': primary_interval,
                        'primary_balanced_accuracy': float(primary_rec.balanced_accuracy),
                        'primary_q_fdr_bh': float(primary_rec.primary_q_fdr_bh),
                        'cross_feature': query_target,
                        'cross_interval': query_interval,
                        'cross_balanced_accuracy': float(cross_rec.balanced_accuracy),
                        'cross_q_fdr_bh': float(cross_rec.feature_matrix_q_fdr_bh),
                    })

    overlap_df = pd.DataFrame(overlap_rows)
    if not overlap_df.empty:
        overlap_df['q_fisher_fdr_bh'] = np.nan
        for assembly, idx in overlap_df.groupby('assembly').groups.items():
            idx = list(idx)
            overlap_df.loc[idx, 'q_fisher_fdr_bh'] = _bh_fdr(overlap_df.loc[idx, 'p_fisher_greater'].to_numpy(dtype=float))
        overlap_df['significant_fisher_fdr_bh_0_05'] = overlap_df['q_fisher_fdr_bh'] < 0.05

    return overlap_df, pd.DataFrame(joint_session_rows)


def plot_joint_cross_feature_heatmap(joint_rows: pd.DataFrame) -> None:
    if joint_rows.empty:
        print('No rows available for the joint cross-feature heatmap.')
        return

    feature_order = [
        (target_name, interval_name)
        for interval_name in JOINT_FEATURE_INTERVALS
        for target_name in JOINT_FEATURE_TARGETS
    ]
    x_labels = [_joint_feature_label(target_name, interval_name) for target_name, interval_name in feature_order]
    assemblies = [assembly for assembly in JOINT_FEATURE_ASSEMBLIES if assembly in set(joint_rows['assembly'].astype(str))]
    fig = make_subplots(
        rows=1,
        cols=len(assemblies),
        subplot_titles=assemblies,
        horizontal_spacing=0.06,
        shared_yaxes=False,
    )

    max_abs = max(0.02, float(np.nanmax(np.abs(joint_rows['shuffle_effect']))) if np.isfinite(joint_rows['shuffle_effect']).any() else 0.02)
    marker_symbols = {
        'primary_significant': 'circle-open',
        'joint_non_relevant': 'circle',
    }

    for col_idx, assembly in enumerate(assemblies, start=1):
        assembly_df = joint_rows[joint_rows['assembly'].astype(str) == assembly].copy()
        primary_sessions = sorted(
            assembly_df.loc[assembly_df['primary_significant'], 'session_id'].astype(str).unique().tolist(),
            key=_session_sort_key,
        )
        if not primary_sessions:
            primary_sessions = sorted(assembly_df['session_id'].astype(str).unique().tolist(), key=_session_sort_key)
        y_labels = [session[:16] for session in primary_sessions]
        y_lookup = dict(zip(primary_sessions, y_labels))
        pair_lookup = {
            (str(rec.session_id), str(rec.target_name), str(rec.interval_name)): rec
            for rec in assembly_df.itertuples(index=False)
        }

        z = np.full((len(primary_sessions), len(feature_order)), np.nan, dtype=float)
        custom = np.full((len(primary_sessions), len(feature_order), 3), np.nan, dtype=float)
        for session_idx, session_id in enumerate(primary_sessions):
            for feature_idx, (target_name, interval_name) in enumerate(feature_order):
                rec = pair_lookup.get((session_id, target_name, interval_name))
                if rec is None:
                    continue
                z[session_idx, feature_idx] = float(rec.shuffle_effect)
                custom[session_idx, feature_idx, :] = [
                    float(rec.balanced_accuracy),
                    float(rec.p_shuffle),
                    float(rec.feature_matrix_q_fdr_bh),
                ]

        fig.add_trace(
            go.Heatmap(
                z=z,
                x=x_labels,
                y=y_labels,
                customdata=custom,
                coloraxis='coloraxis',
                xgap=1,
                ygap=1,
                hovertemplate=(
                    'feature x interval=%{x}<br>session=%{y}<br>'
                    'shuffle effect=%{z:.4f}<br>'
                    'balanced accuracy=%{customdata[0]:.3f}<br>'
                    'p shuffle=%{customdata[1]:.4g}<br>'
                    'focused BH-FDR q=%{customdata[2]:.4g}<extra></extra>'
                ),
            ),
            row=1,
            col=col_idx,
        )

        primary_marker_x = []
        primary_marker_y = []
        joint_marker_x = []
        joint_marker_y = []
        primary_session_set = set(primary_sessions)
        for rec in assembly_df.itertuples(index=False):
            session_id = str(rec.session_id)
            if session_id not in primary_session_set:
                continue
            x_label = _joint_feature_label(str(rec.target_name), str(rec.interval_name))
            y_label = y_lookup[session_id]
            if bool(rec.primary_significant):
                primary_marker_x.append(x_label)
                primary_marker_y.append(y_label)
            if bool(rec.non_relevant_significant):
                joint_marker_x.append(x_label)
                joint_marker_y.append(y_label)

        if primary_marker_x:
            fig.add_trace(
                go.Scatter(
                    x=primary_marker_x,
                    y=primary_marker_y,
                    mode='markers',
                    marker={'symbol': marker_symbols['primary_significant'], 'size': 8, 'color': 'black', 'line': {'width': 1.4}},
                    name='primary q<0.05',
                    legendgroup='primary',
                    showlegend=col_idx == 1,
                    hovertemplate='primary significant<br>%{x}<br>%{y}<extra></extra>',
                ),
                row=1,
                col=col_idx,
            )
        if joint_marker_x:
            fig.add_trace(
                go.Scatter(
                    x=joint_marker_x,
                    y=joint_marker_y,
                    mode='markers',
                    marker={'symbol': marker_symbols['joint_non_relevant'], 'size': 7, 'color': 'black', 'line': {'width': 0.8, 'color': 'white'}},
                    name='non-relevant q<0.05',
                    legendgroup='joint',
                    showlegend=col_idx == 1,
                    hovertemplate='non-relevant significant<br>%{x}<br>%{y}<extra></extra>',
                ),
                row=1,
                col=col_idx,
            )

        for boundary_after in [2, 5]:
            fig.add_vline(x=boundary_after + 0.5, line={'color': 'black', 'width': 1.2}, row=1, col=col_idx)
        fig.update_xaxes(tickangle=0, tickfont={'size': 7}, showline=True, linecolor='black', mirror=True, row=1, col=col_idx)
        fig.update_yaxes(title_text='session' if col_idx == 1 else None, tickfont={'size': 7}, showline=True, linecolor='black', mirror=True, autorange='reversed', row=1, col=col_idx)

    fig.update_layout(
        template='plotly_white+pub',
        title={'text': 'Joint cross-feature decoding in primary-significant sessions', 'x': 0.02, 'xanchor': 'left', 'font': {'size': 12, 'color': 'black'}},
        font={'family': 'Arial, Helvetica, sans-serif', 'size': 8, 'color': 'black'},
        coloraxis={
            'colorscale': 'RdBu_r',
            'cmin': -max_abs,
            'cmax': max_abs,
            'cmid': 0.0,
            'colorbar': {'title': {'text': 'Shuffle effect', 'font': {'size': 9}}, 'thickness': 10, 'len': 0.68, 'tickfont': {'size': 8}},
        },
        legend={'orientation': 'v', 'x': 1.02, 'xanchor': 'left', 'y': 1.0, 'yanchor': 'top', 'font': {'size': 8}, 'itemsizing': 'constant'},
        width=930,
        height=max(330, 26 * max(joint_rows.groupby('assembly')['session_id'].nunique()) + 160),
        margin={'l': 86, 'r': 156, 't': 68, 'b': 58},
        paper_bgcolor='white',
        plot_bgcolor='white',
    )
    fig.show()


joint_cross_feature_rows_df = prepare_joint_cross_feature_rows(session_results_df)
joint_overlap_tests_df, joint_cross_feature_sessions_df = compute_joint_cross_feature_tables(joint_cross_feature_rows_df)

if joint_cross_feature_rows_df.empty:
    print('No joint cross-feature rows were available after filtering.')
else:
    print('Primary-significant sessions by assembly:')
    for assembly, assembly_df in joint_cross_feature_rows_df.groupby('assembly', sort=False):
        primary_sessions = sorted(assembly_df.loc[assembly_df['primary_significant'], 'session_id'].astype(str).unique().tolist(), key=_session_sort_key)
        print(f'  {assembly}: {primary_sessions}')

    print('\nJoint non-relevant encodings in primary-significant sessions:')
    if joint_cross_feature_sessions_df.empty:
        print('  None detected after focused BH-FDR correction.')
    else:
        display(joint_cross_feature_sessions_df.sort_values(['assembly', 'session_id', 'primary_interval', 'cross_interval', 'cross_feature']))

    if not joint_overlap_tests_df.empty:
        display(
            joint_overlap_tests_df.sort_values(['q_fisher_fdr_bh', 'p_fisher_greater', 'assembly'])[
                [
                    'assembly', 'primary_feature', 'primary_interval', 'cross_feature', 'cross_interval',
                    'n_eligible_sessions', 'n_primary_significant', 'n_cross_significant', 'n_joint_sessions',
                    'p_fisher_greater', 'q_fisher_fdr_bh', 'significant_fisher_fdr_bh_0_05', 'joint_sessions',
                ]
            ]
        )

    plot_joint_cross_feature_heatmap(joint_cross_feature_rows_df)


Cross-feature decoding across intervals was sparse in the two highlighted ensembles. In Assembly007, sessions with interval-relevant decoding did not show BH-FDR-significant decoding of non-relevant feature/interval combinations. In Assembly023, only session `2024-12-09_17-45` showed joint encoding: cue information was significant during the R1-entry interval, while the same session also showed interval-relevant R1-choice decoding at R1 entry and R2-choice decoding at R2 entry. However, the Fisher overlap tests were not significant after correction, indicating a session-specific joint representation rather than a consistent cross-interval encoding motif across the ensemble.
